# Observabilidade em ML e Atualização Online de Modelos
### Demonstração: modelo de detecção de fraude em cartão de crédito

**A história:** um banco usa um modelo para bloquear transações suspeitas.

| Ato | O que acontece | Tipo de mudança |
|---|---|---|
| 1. Dia normal | Tudo funciona | — |
| 2. Black Friday | Todo mundo compra mais, mais caro e mais rápido | **Data drift** (os dados mudam, a fraude não) |
| 3. Golpistas mudam de tática | "Teste de cartão": valores minúsculos, madrugada, várias tentativas | **Concept drift** (a própria fraude muda) |

Dois modelos recebem **exatamente as mesmas transações**:
- 🔴 **Estático:** treinado uma vez e nunca mais atualizado.
- 🟢 **Online:** aprende com cada lote assim que os rótulos (chargebacks) chegam.

> ⚠️ Os dados são **simulados** (didáticos). Rode as células de cima para baixo (`Shift+Enter`).

## 1. Configuração

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
from sklearn.linear_model import SGDClassifier

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

SEED = 11
FEATURES = ["log_valor", "madrugada", "log_dist_km", "tx_na_hora"]
BATCH = 250                    # transações por hora
N_STEPS = 150                  # horas simuladas
BF_START, BF_END = 50, 90      # Black Friday
NEW_FRAUD_AT = 90              # golpistas mudam de tática
LABEL_DELAY = 4                # o chargeback chega 4 horas depois
WINDOW = 4
PSI_LIMIT = 0.20               # PSI > 0.2 = "o perfil dos dados mudou"
RECALL_SLO = 0.70              # meta mínima: capturar 70% das fraudes
BLOCK_RATE_FACTOR = 1.8
FP_COST = 30.0                 # R$ por cliente bom bloqueado
FRAUD_FIXED_COST = 120.0       # R$ de chargeback + operação por fraude que passou
LR_NORMAL, LR_BOOST, BOOST_STEPS = 0.01, 0.06, 10

C_S, C_O = "#e74c3c", "#16a085"   # vermelho = estático, verde = online
ACTS = [(0, BF_START, "Ato 1: dia normal", "#2ecc71"),
        (BF_START, BF_END, "Ato 2: Black Friday", "#f39c12"),
        (NEW_FRAUD_AT, N_STEPS, "Ato 3: nova fraude", "#e74c3c")]

## 2. Os dados (simulados)
Cada transação tem 4 características: **valor** (em log), se foi de **madrugada**, **distância** de casa (em log) e **quantas transações** o cartão fez na última hora.

In [ ]:
class Bank:
    def __init__(self, rng):
        self.rng = rng

    def legit(self, n, black_friday):
        r = self.rng
        lv = r.normal(4.0 + (0.8 if black_friday else 0.0), 0.8, n)
        mad = r.random(n) < (0.18 if black_friday else 0.05)
        ld = r.normal(1.0, 0.8, n)
        tx = r.poisson(2.6 if black_friday else 1.0, n)
        return np.column_stack([lv, mad, ld, tx])

    def fraud_classic(self, n):
        r = self.rng
        return np.column_stack([r.normal(6.0, 0.8, n), r.random(n) < 0.5,
                                r.normal(3.0, 0.8, n), r.poisson(3.0, n)])

    def fraud_card_testing(self, n):          # a fraude NOVA do Ato 3
        r = self.rng
        return np.column_stack([r.normal(2.0, 0.5, n), r.random(n) < 0.85,
                                r.normal(1.2, 0.8, n), r.poisson(5.0, n)])

    def batch(self, step, n=BATCH):
        bf = BF_START <= step < BF_END
        new = step >= NEW_FRAUD_AT
        n_fraud = self.rng.binomial(n, 0.08 if new else 0.06)
        Xl = self.legit(n - n_fraud, bf)
        Xf = self.fraud_card_testing(n_fraud) if new else self.fraud_classic(n_fraud)
        X = np.vstack([Xl, Xf]).astype(float)
        y = np.concatenate([np.zeros(n - n_fraud), np.ones(n_fraud)]).astype(int)
        idx = self.rng.permutation(n)
        return X[idx], y[idx]

**Olhe a diferença entre os grupos:** o cliente bom muda na Black Friday (laranja) e a fraude nova (vermelho) tem valor *menor* que o normal. É isso que confunde o modelo antigo.

In [ ]:
b = Bank(np.random.default_rng(0))
grupos = {"Cliente normal": b.legit(4000, False),
          "Cliente na Black Friday": b.legit(4000, True),
          "Fraude clássica": b.fraud_classic(1000),
          "Fraude nova (teste de cartão)": b.fraud_card_testing(1000)}
cores = ["#2ecc71", "#f39c12", "#3498db", "#e74c3c"]

fig, axs = plt.subplots(1, 2, figsize=(12, 3.6))
for (nome, X), c in zip(grupos.items(), cores):
    axs[0].hist(np.exp(X[:, 0]), bins=np.linspace(0, 1500, 60), alpha=0.55, color=c, label=nome, density=True)
    axs[1].hist(X[:, 3], bins=np.arange(-0.5, 12.5, 1), alpha=0.55, color=c, label=nome, density=True)
axs[0].set_title("Valor da compra (R$)"); axs[0].legend(fontsize=8)
axs[1].set_title("Transações na última hora")
plt.tight_layout(); plt.show()

## 3. Os dois modelos
Mesmo algoritmo (regressão logística treinada com SGD), mesmo treino inicial **só com dados de um dia normal**.
A única diferença: o **online** usa `partial_fit` para continuar aprendendo; o **estático** nunca mais é tocado.

In [ ]:
rng = np.random.default_rng(SEED)
bank = Bank(rng)
tr = [bank.batch(0) for _ in range(80)]
X_tr = np.vstack([t[0] for t in tr]); y_tr = np.concatenate([t[1] for t in tr])
mu, sd = X_tr.mean(0), X_tr.std(0) + 1e-9
scale = lambda X: (X - mu) / sd

def make_model():
    m = SGDClassifier(loss="log_loss", learning_rate="constant", eta0=LR_NORMAL,
                      alpha=1e-4, class_weight={0: 1, 1: 4}, random_state=SEED)
    m.fit(scale(X_tr), y_tr)
    return m

m = make_model()
print(f"Treino com {len(y_tr)} transações normais ({y_tr.mean()*100:.1f}% fraudes)")
print(f"Taxa de bloqueio normal: {m.predict(scale(X_tr)).mean()*100:.1f}%")

## 4. Ferramenta de observabilidade nº 1: o PSI
O **PSI (Population Stability Index)** compara a distribuição de uma característica *hoje* com a do *treino*.
Regra de bolso: **< 0,1** estável · **0,1–0,2** atenção · **> 0,2** mudou. **Não precisa de rótulo.**

In [ ]:
def psi(ref, cur, bins=10):
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    p = np.histogram(ref, edges)[0] / len(ref)
    q = np.histogram(cur, edges)[0] / len(cur)
    p, q = np.clip(p, 1e-4, None), np.clip(q, 1e-4, None)
    return float(np.sum((q - p) * np.log(q / p)))

ref = np.random.default_rng(1).normal(0, 1, 5000)
for desc, cur in [("Mesma distribuição", np.random.default_rng(2).normal(0, 1, 1000)),
                  ("Média deslocada um pouco (+0.5)", np.random.default_rng(3).normal(0.5, 1, 1000)),
                  ("Média deslocada muito (+1.5)", np.random.default_rng(4).normal(1.5, 1, 1000))]:
    print(f"{desc:35s} -> PSI = {psi(ref, cur):.2f}")

## 5. A simulação: 150 horas de transações
A cada hora, para cada modelo: **prever → medir sinais → esperar o rótulo → (só o online) aprender.**

Os sinais acompanhados:
- **Taxa de bloqueio** e **PSI** → *não precisam de rótulo* (rápidos)
- **Recall** (fraudes capturadas) → *precisa de rótulo*, que chega com **atraso** (lento, mas é a verdade)
- **Prejuízo em R$** → métrica de negócio

Quando algo dispara, o sistema **age sozinho**: o online passa a aprender mais rápido por um tempo.

In [ ]:
def act_name(step):
    return next(n for s, e, n, _ in ACTS if s <= step < e)

def recall_of(win):
    tp = sum(a for a, _ in win); fn = sum(b for _, b in win)
    return tp / (tp + fn) if (tp + fn) else np.nan

def simular(label_delay=LABEL_DELAY, online_learns=True, verbose=True):
    rng = np.random.default_rng(SEED); bank = Bank(rng)
    for _ in range(80): bank.batch(0)            # mesmo estado do treino
    static, online = make_model(), make_model()
    base_block = float(online.predict(scale(X_tr)).mean())

    pending, win_X, blk_win = deque(), deque(maxlen=WINDOW), deque(maxlen=WINDOW)
    rec_win = {"s": deque(maxlen=WINDOW), "o": deque(maxlen=WINDOW)}
    H = {k: [] for k in ["t", "bs", "bo", "rs", "ro", "cs", "co"]}; H["psi"] = [[] for _ in FEATURES]
    cost = {"s": 0.0, "o": 0.0}; events = []; first = {}
    state = {"psi": False, "rec": False, "blk": False}; boost_left = 0

    def event(step, text):
        events.append((step, text))
        if verbose: print(f"[hora {step:3d}] {text}")

    for step in range(N_STEPS):
        X, y = bank.batch(step); Z = scale(X)
        ps, po = static.predict(Z), online.predict(Z)
        valor = np.exp(X[:, 0])

        # prejuízo (o simulador conhece a verdade; o banco só descobre depois)
        for tag, p in (("s", ps), ("o", po)):
            miss = (y == 1) & (p == 0)
            cost[tag] += valor[miss].sum() + miss.sum() * FRAUD_FIXED_COST + ((y == 0) & (p == 1)).sum() * FP_COST

        # sinais SEM rótulo
        blk_win.append(po.mean()); win_X.append(X)
        Xw = np.vstack(win_X)
        psis = [psi(X_tr[:, j], Xw[:, j]) for j in range(len(FEATURES))]

        # rótulos chegam com atraso -> mede recall e (só o online) aprende
        pending.append((step + label_delay, Z, y, ps, po))
        while pending and pending[0][0] <= step:
            _, Zl, yl, psl, pol = pending.popleft(); pos = yl == 1
            rec_win["s"].append(((psl[pos] == 1).sum(), (psl[pos] == 0).sum()))
            rec_win["o"].append(((pol[pos] == 1).sum(), (pol[pos] == 0).sum()))
            if online_learns: online.partial_fit(Zl, yl)
        rec_s, rec_o = recall_of(rec_win["s"]), recall_of(rec_win["o"])

        # alertas (sobre o modelo em produção = online)
        worst = int(np.argmax(psis))
        psi_bad = max(psis) > PSI_LIMIT and step >= WINDOW
        ratio = float(np.mean(blk_win)) / base_block
        blk_bad = step >= WINDOW and (ratio > BLOCK_RATE_FACTOR or ratio < 1 / BLOCK_RATE_FACTOR)
        rec_bad = (not np.isnan(rec_o)) and rec_o < RECALL_SLO
        new_alert = False
        if psi_bad and not state["psi"]:
            event(step, f"ALERTA: perfil dos dados mudou ({FEATURES[worst]}, PSI={max(psis):.2f})"); first.setdefault("psi", step); new_alert = True
        if not psi_bad and state["psi"]: event(step, "OK: perfil dos dados voltou ao normal")
        if blk_bad and not state["blk"]:
            event(step, f"ALERTA: taxa de bloqueio anormal ({np.mean(blk_win)*100:.0f}% vs {base_block*100:.0f}%)"); first.setdefault("blk", step); new_alert = True
        if not blk_bad and state["blk"]: event(step, "OK: taxa de bloqueio normalizada")
        if rec_bad and not state["rec"]:
            event(step, f"CRÍTICO: recall {rec_o*100:.0f}% abaixo da meta de {RECALL_SLO*100:.0f}%"); first.setdefault("rec", step); new_alert = True
        if (not rec_bad) and state["rec"] and not np.isnan(rec_o) and rec_o > RECALL_SLO + 0.1:
            event(step, f"OK: recall recuperado ({rec_o*100:.0f}%)"); state["rec"] = False
        if rec_bad: state["rec"] = True
        state["psi"], state["blk"] = psi_bad, blk_bad

        if new_alert and boost_left == 0:
            boost_left = BOOST_STEPS; online.set_params(eta0=LR_BOOST)
            event(step, f"AÇÃO: modelo online aprende mais rápido (lr {LR_NORMAL} -> {LR_BOOST})")
        elif boost_left > 0:
            boost_left -= 1
            if boost_left == 0:
                online.set_params(eta0=LR_NORMAL); event(step, "AÇÃO: aprendizado volta ao ritmo normal")

        H["t"].append(step); H["bs"].append(ps.mean() * 100); H["bo"].append(po.mean() * 100)
        H["rs"].append(rec_s); H["ro"].append(rec_o)
        H["cs"].append(cost["s"] / 1000); H["co"].append(cost["o"] / 1000)
        for j in range(len(FEATURES)): H["psi"][j].append(psis[j])
    return dict(H=H, cost=cost, events=events, first=first)

R = simular()

## 6. Os gráficos: como ler cada um

In [ ]:
def fundo(ax):
    for s, e, _, col in ACTS: ax.axvspan(s, e, color=col, alpha=0.10, lw=0)
    for s, e, nome, _ in ACTS: ax.text((s + e) / 2, 1.01, nome, transform=ax.get_xaxis_transform(), ha="center", fontsize=8)
    ax.set_xlim(0, N_STEPS); ax.set_xlabel("tempo (horas)")
H = R["H"]

### Gráfico 1 — Quanto estamos bloqueando? *(sem rótulo → sinal rápido)*
De cada 100 compras, quantas foram bloqueadas. Normal ≈ **7%**.
- **Black Friday:** o 🔴 estático sobe para ~20% (bloqueia clientes bons). O 🟢 online volta ao normal.
- **Ato 3:** o 🔴 cai para ~0%: **parou de bloquear** e deixa a fraude passar.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8)); fundo(ax)
ax.plot(H["t"], H["bs"], color=C_S, lw=2, label="Estático"); ax.plot(H["t"], H["bo"], color=C_O, lw=2, label="Online")
ax.axhline(7, color="k", ls=":", lw=1); ax.text(1, 8.5, "normal ≈ 7%", fontsize=8)
ax.set_ylabel("% bloqueadas"); ax.legend(loc="upper left"); plt.show()

### Gráfico 2 — Fraudes capturadas, o *recall* *(com rótulo → lento, mas é a verdade)*
De cada 100 fraudes, quantas o modelo pegou. A linha amarela é a **meta mínima (SLO) de 70%**.
- No Ato 3 os dois **despencam** (a fraude é nova). O 🔴 estático **nunca se recupera**. O 🟢 online **se recupera sozinho** em poucas horas.
- Note que a queda só aparece **depois** que os chargebacks chegam.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8)); fundo(ax)
ax.plot(H["t"], H["rs"], color=C_S, lw=2, label="Estático"); ax.plot(H["t"], H["ro"], color=C_O, lw=2, label="Online")
ax.axhline(RECALL_SLO, color="#d4a017", ls="--"); ax.text(1, RECALL_SLO + 0.02, "meta mínima (SLO) 70%", fontsize=8, color="#d4a017")
ax.set_ylabel("recall"); ax.set_ylim(0, 1.05); ax.legend(loc="lower left"); plt.show()

### Gráfico 3 — Mudou o perfil dos dados? *PSI (sem rótulo → sinal rápido)*
Acima da linha amarela (0,2) = o perfil dos dados mudou.
- **Black Friday:** dispara forte.
- **Ato 3:** **quase não se mexe** — a fraude é só 8% das compras. *Por isso o PSI sozinho não basta: precisamos também do recall.*

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8)); fundo(ax)
for j, f in enumerate(FEATURES): ax.plot(H["t"], H["psi"][j], lw=1.8, label=f)
ax.axhline(PSI_LIMIT, color="#d4a017", ls="--"); ax.set_ylabel("PSI"); ax.legend(loc="upper left", ncol=2, fontsize=8); plt.show()

### Gráfico 4 — Prejuízo acumulado *(a métrica de negócio)*
Fraude que passou (valor + R$ 120 de chargeback) + clientes bons bloqueados (R$ 30 cada).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8)); fundo(ax)
ax.plot(H["t"], H["cs"], color=C_S, lw=2.4, label="Estático"); ax.plot(H["t"], H["co"], color=C_O, lw=2.4, label="Online")
ax.set_ylabel("R$ mil"); ax.legend(loc="upper left")
ax.text(0.98, 0.05, f"Estático: R$ {R['cost']['s']/1000:,.0f} mil\nOnline: R$ {R['cost']['o']/1000:,.0f} mil",
        transform=ax.transAxes, ha="right", fontweight="bold"); plt.show()

## 7. Resumo

In [ ]:
c = R["cost"]; eco = c["s"] - c["o"]
print(f"Prejuízo acumulado | estático: R$ {c['s']:,.0f}  |  online: R$ {c['o']:,.0f}")
print(f"Economia por atualizar online: R$ {eco:,.0f} ({eco / c['s'] * 100:.0f}%)")
for k, nome in (("psi", "mudança de perfil (PSI)"), ("blk", "taxa de bloqueio"), ("rec", "queda do recall")):
    if k in R["first"]: print(f"Primeiro alerta de {nome}: hora {R['first'][k]}")

> **Lição:** a **observabilidade avisa** que algo mudou; a **atualização online corrige**. Sinais *sem rótulo* (PSI, taxa de bloqueio) são rápidos mas incompletos; sinais *com rótulo* (recall) são a verdade, mas chegam atrasados. Precisamos dos dois.

## 8. (Opcional) Experimentos — "e se...?"
Boa hora no vídeo para mostrar que o resultado depende das escolhas:

In [ ]:
# E se o modelo online NÃO aprendesse (só observabilidade, sem atualização)?
R0 = simular(online_learns=False, verbose=False)
print(f"Só observabilidade, sem atualizar: R$ {R0['cost']['o']:,.0f}   (alertas disparam, mas ninguém corrige)")

# E se os rótulos demorassem muito mais (chargeback em 12 horas)?
R12 = simular(label_delay=12, verbose=False)
print(f"Rótulos com 12h de atraso:        R$ {R12['cost']['o']:,.0f}   (corrige, mas mais devagar)")
print(f"Referência (atraso 4h):           R$ {R['cost']['o']:,.0f}")

## 9. (Opcional) Reprodução "ao vivo" do dashboard
Execute para ver o dashboard sendo desenhado hora a hora, como no vídeo da demonstração.

In [ ]:
from IPython.display import clear_output
import time

def ao_vivo(passo=3, pausa=0.03):
    for n in list(range(passo, N_STEPS, passo)) + [N_STEPS]:
        fig, axs = plt.subplots(2, 2, figsize=(12, 6.2))
        t = H["t"][:n]
        for ax in axs.flat: fundo(ax)
        axs[0, 0].plot(t, H["bs"][:n], color=C_S); axs[0, 0].plot(t, H["bo"][:n], color=C_O); axs[0, 0].set_title("Taxa de bloqueio (%)", fontsize=10); axs[0, 0].set_ylim(0, 60)
        axs[0, 1].plot(t, H["rs"][:n], color=C_S); axs[0, 1].plot(t, H["ro"][:n], color=C_O); axs[0, 1].axhline(RECALL_SLO, color="#d4a017", ls="--"); axs[0, 1].set_title("Recall (fraudes capturadas)", fontsize=10); axs[0, 1].set_ylim(0, 1.05)
        for j in range(len(FEATURES)): axs[1, 0].plot(t, H["psi"][j][:n], lw=1.5)
        axs[1, 0].axhline(PSI_LIMIT, color="#d4a017", ls="--"); axs[1, 0].set_title("PSI (mudança nos dados)", fontsize=10); axs[1, 0].set_ylim(0, 2.5)
        axs[1, 1].plot(t, H["cs"][:n], color=C_S, label="Estático"); axs[1, 1].plot(t, H["co"][:n], color=C_O, label="Online"); axs[1, 1].set_title("Prejuízo acumulado (R$ mil)", fontsize=10); axs[1, 1].legend(loc="upper left")
        fig.suptitle(act_name(n - 1), fontweight="bold"); fig.tight_layout()
        clear_output(wait=True); plt.show(); time.sleep(pausa)

ao_vivo()